# Example 2: Intermediate Globe (Coastline step & Hemisphere splitting)

This notebook demonstrates how to create a more advanced globe that combines standard topographic displacement with a sharp, physical step boundary at the coastlines (loaded from a shapefile). We then split the displaced model into capped top and bottom hemispheres, making them easy to print flat on a build plate.

## Step 1: Import libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import trimesh
from globe3d import (
    generate_sphere_points_fibonacci,
    load_netcdf_grid,
    calculate_displacement_scale,
    displace_vertices,
    displace_near_lines,
    split_mesh_hemispheres
)

## Step 2: Generate base sphere and apply topography

We generate an 8,000-point Fibonacci sphere and apply ETOPO grid elevation displacement.

In [ ]:
model_radius_mm = 40.0
vertices, faces = generate_sphere_points_fibonacci(n_points=8000, radius=model_radius_mm)

netcdf_path = "../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc"
lats, lons, grid = load_netcdf_grid(netcdf_path, 'lat', 'lon', 'z')
lats_ds, lons_ds, grid_ds = lats[::10], lons[::10], grid[::10, ::10]

scale = calculate_displacement_scale(model_radius_mm, earth_radius_km=6371.0, vertical_exagg=40.0)
vertices = displace_vertices(vertices, lats_ds, lons_ds, grid_ds, scale)

## Step 3: Apply coastline step boundary

We use a Natural Earth coastline shapefile to identify vertices close to the shorelines and displace them by an extra 0.8 mm. This creates a clean tactile ledge at the coast on the physical print.

In [ ]:
coastline_shp = "../inputs/coastlines/ne_110m_coastline.shp"

vertices = displace_near_lines(
    vertices=vertices,
    shapefile_path=coastline_shp,
    displacement=0.8,   # 0.8 mm step height
    width_degrees=0.5   # 0.5 degrees ribbon width
)
print("Applied coastline step displacement.")

## Step 4: Create a watertight Trimesh object

We build a Trimesh object and fix normal directions to ensure the mesh has positive volume.

In [ ]:
mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
mesh.fix_normals()
print(f"Mesh is watertight: {mesh.is_watertight}")

## Step 5: Split the mesh into capped hemispheres

We split the mesh along the equator (XY plane). Both halves are capped to create flat solid faces.

In [ ]:
top_half, bottom_half = split_mesh_hemispheres(mesh, normal=(0, 0, 1), origin=(0, 0, 0))
print(f"Top hemisphere is watertight: {top_half.is_watertight}")
print(f"Bottom hemisphere is watertight: {bottom_half.is_watertight}")

## Step 6: Preview both hemispheres

In [ ]:
fig = plt.figure(figsize=(12, 6))

# Top half
ax1 = fig.add_subplot(121, projection='3d')
pts_top = top_half.vertices
sc1 = ax1.scatter(pts_top[:, 0], pts_top[:, 1], pts_top[:, 2], c=pts_top[:, 2], cmap='viridis', s=2)
ax1.set_title("Top Hemisphere")

# Bottom half
ax2 = fig.add_subplot(122, projection='3d')
pts_bot = bottom_half.vertices
sc2 = ax2.scatter(pts_bot[:, 0], pts_bot[:, 1], pts_bot[:, 2], c=pts_bot[:, 2], cmap='viridis', s=2)
ax2.set_title("Bottom Hemisphere")

plt.show()

## Step 7: Export STL files

In [ ]:
output_dir = "../outputs"
os.makedirs(output_dir, exist_ok=True)

top_half.export(os.path.join(output_dir, "example_2_top.stl"))
bottom_half.export(os.path.join(output_dir, "example_2_bottom.stl"))
print("Hemispheres exported to outputs/")